# Step 0 — Environment & Model Setup Verification

This notebook verifies:
1. Loading environment variables from `.env` (`GROQ_API_KEY`, `TAVILY_API_KEY`, `MONGODB_URI`).
2. MongoDB Atlas database connection via `pymongo`.
3. Declarations and test invocations of `light_model` and `deep_model` using `langchain-groq`.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

groq_key = os.getenv('GROQ_API_KEY')
tavily_key = os.getenv('TAVILY_API_KEY')
mongo_uri = os.getenv('MONGODB_URI')

print('GROQ_API_KEY present:', bool(groq_key))
print('TAVILY_API_KEY present:', bool(tavily_key))
print('MONGODB_URI present:', bool(mongo_uri))

## 1. Verify MongoDB Connection

In [ ]:
from pymongo import MongoClient

if not mongo_uri:
    print('⚠️ MONGODB_URI is not set in .env! Please add your connection string to .env file.')
else:
    try:
        client = MongoClient(mongo_uri)
        dbs = client.list_database_names()
        print('✅ MongoDB Connection Successful!')
        print('Available Databases:', dbs)
    except Exception as e:
        print('❌ MongoDB Connection Failed:', e)

## 2. Declare and Verify LLM Models (Light Model & Deep Model)

We follow the exact project model declaration pattern below.

In [ ]:
# ============================================================
# LIGHT MODEL — simple, low-reasoning calls (e.g. RAG answer
# synthesis from already-retrieved context, memory summarization)
# CURRENT: Groq, free tier, no card required
# TO REPLACE WITH A DIFFERENT FREE MODEL LATER:
#   1. pip install the new provider's langchain package
#   2. change the import below
#   3. change the class + model name below
#   4. add the new provider's key name to .env (keep GROQ_API_KEY
#      too if other files still use Groq)
# ============================================================
from langchain_groq import ChatGroq
light_model = ChatGroq(model_name='llama-3.3-70b-versatile', temperature=0)


# ============================================================
# DEEP MODEL — multi-step reasoning (e.g. dispatcher tool
# routing/chaining, compare-tool pymongo code generation)
# CURRENT: Groq, free tier (same as light model — no dedicated
# "deep thinking" free model chosen yet)
# TO REPLACE WITH A DEEP-THINKING MODEL WHEN YOU FIND ONE:
#   1. pip install the new provider's langchain package
#      (e.g. langchain-google-genai for a Gemini thinking model)
#   2. change the import below
#   3. change the class + model name below (some providers add
#      extra params here, e.g. Gemini's thinking_budget=... —
#      check that provider's LangChain docs for what's available)
#   4. add the new provider's key name to .env
# ============================================================
from langchain_groq import ChatGroq
deep_model = ChatGroq(model_name='llama-3.3-70b-versatile', temperature=0.2)

print('Light model & Deep model initialized successfully.')

In [ ]:
if not groq_key:
    print('⚠️ GROQ_API_KEY is not set in .env! Please set GROQ_API_KEY in .env to test model invocations.')
else:
    print('Testing Light Model Invocation...')
    res_light = light_model.invoke('say hi')
    print('Light Model Response:', res_light.content)
    
    print('\nTesting Deep Model Invocation...')
    res_deep = deep_model.invoke('say hi')
    print('Deep Model Response:', res_deep.content)